In [1]:
import os
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
from pyspark.sql import functions as F
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .appName("projectTest")
    .master("local[*]")
    .config("spark.ui.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark UI: http://localhost:4040


In [2]:
# Ler o ficheiro audio_features.csv e billboard.csv
df_af = spark.read.csv("C:\\Users\\lifan\\BigData\\projeto\\data\\audio_features.csv", header=True, inferSchema=True, sep=",")
df_bb = spark.read.csv("C:\\Users\\lifan\\BigData\\projeto\\data\\billboard.csv", header=True, inferSchema=True, sep=",")

In [ ]:
# Exploração de dados do ficheiro audio_features.csv

In [3]:
# Mostrar o esquema do DataFrame do ficheiro audio_features.csv, assim visualizando os tipos de dados de cada coluna
df_af.printSchema()


root
 |-- song_id: string (nullable = true)
 |-- performer: string (nullable = true)
 |-- song: string (nullable = true)
 |-- spotify_genre: string (nullable = true)
 |-- spotify_track_id: string (nullable = true)
 |-- spotify_track_preview_url: string (nullable = true)
 |-- spotify_track_duration_ms: string (nullable = true)
 |-- spotify_track_explicit: string (nullable = true)
 |-- spotify_track_album: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- key: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- valence: string (nullable = true)
 |-- tempo: string (nullable = true)
 |-- time_signature: string (nullable = true)
 |-- spotify_track_popularity: string (nullable = true)



In [4]:
# contar o número de linhas do DataFrame do ficheiro audio_features.csv
df_af.count()

29503

In [5]:
# contar o número de colunas (total de variáveis) do ficheiro audio_features.csv
len(df_af.columns)

22

In [6]:
# Visualizar as primeiras 10 linhas do DataFrame do ficheiro audio_features.csv
df_af.show(10)


+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------------+-------------------------+----------------------+--------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+------------------------+
|             song_id|           performer|                song|       spotify_genre|    spotify_track_id|spotify_track_preview_url|spotify_track_duration_ms|spotify_track_explicit| spotify_track_album|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|spotify_track_popularity|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------------+-------------------------+----------------------+--------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----

In [7]:
# Verificar se existem linhas duplicadas no DataFrame do ficheiro audio_features.csv
duplicates = (
               df_af
              .groupBy(df_af.columns)
              .count()
              .filter("count > 1")
              )
duplicates.show()
duplicates.count()

+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------------+-------------------------+----------------------+--------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+------------------------+-----+
|             song_id|           performer|                song|       spotify_genre|    spotify_track_id|spotify_track_preview_url|spotify_track_duration_ms|spotify_track_explicit| spotify_track_album|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|spotify_track_popularity|count|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------------+-------------------------+----------------------+--------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-

24

In [ ]:
# Contar o número de missing values (null ou vazios) em cada coluna dos variáveis do ficheiro audio_features.csv
missing_counts = df_af.select([
    F.count(F.when((F.col(c).isNull()) | (F.col(c) == ""), c)).alias(c)
    for c in df_af.columns
])

missing_counts.show(truncate=False)

+-------+---------+----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+
|song_id|performer|song|spotify_genre|spotify_track_id|spotify_track_preview_url|spotify_track_duration_ms|spotify_track_explicit|spotify_track_album|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|spotify_track_popularity|
+-------+---------+----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+
|0      |0        |0   |0            |0               |0                        |0                        |0          

In [9]:
# Contar o número de valores "NA" em cada coluna dos variáveis do ficheiro audio_features.csv
na_values = df_af.select([
    F.count(F.when((F.col(c).isNotNull()) & (F.col(c) == "NA"), c)).alias(c)
    for c in df_af.columns
])

na_values.show(truncate=False)

+-------+---------+----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+----+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+
|song_id|performer|song|spotify_genre|spotify_track_id|spotify_track_preview_url|spotify_track_duration_ms|spotify_track_explicit|spotify_track_album|danceability|energy|key |loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|spotify_track_popularity|
+-------+---------+----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+----+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+
|0      |0        |0   |1599         |5097            |15004                    |5105                     |5106    

In [1]:
# Alterar o tipo de dados das colunas numéricas para double
'''
cols_to_double = [
    "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "time_signature",
    "spotify_track_popularity", "spotify_track_duration_ms"
]

for c in cols_to_double:
    df_af = df_af.withColumn(c, F.col(c).cast("double"))
df_af.printSchema()
'''


'\ncols_to_double = [\n    "danceability", "energy", "key", "loudness", "mode",\n    "speechiness", "acousticness", "instrumentalness",\n    "liveness", "valence", "tempo", "time_signature",\n    "spotify_track_popularity", "spotify_track_duration_ms"\n]\n\nfor c in cols_to_double:\n    df_af = df_af.withColumn(c, F.col(c).cast("double"))\ndf_af.printSchema()\n'

In [ ]:
# Exploração de dados do ficheiro billboard.csv

In [ ]:
# Visualizar o esquema do DataFrame do ficheiro billboard.csv, assim visualizando os tipos de dados de cada coluna
df_bb.printSchema()

root
 |-- url: string (nullable = true)
 |-- week_id: string (nullable = true)
 |-- week_position: integer (nullable = true)
 |-- song: string (nullable = true)
 |-- performer: string (nullable = true)
 |-- song_id: string (nullable = true)
 |-- instance: string (nullable = true)
 |-- previous_week_position: string (nullable = true)
 |-- peak_position: string (nullable = true)
 |-- weeks_on_chart: string (nullable = true)



In [ ]:
# contar o número de linhas de dados do ficheiro billboard.csv
df_bb.count()

327895

In [ ]:
# contar o número de colunas (total de variáveis) do ficheiro billboard.csv
len(df_bb.columns)

10

In [ ]:
# Visualizar as primeiras 10 linhas de dados do ficheiro billboard.csv
df_bb.show(10)

+--------------------+---------+-------------+--------------------+-----------------+--------------------+--------+----------------------+-------------+--------------+
|                 url|  week_id|week_position|                song|        performer|             song_id|instance|previous_week_position|peak_position|weeks_on_chart|
+--------------------+---------+-------------+--------------------+-----------------+--------------------+--------+----------------------+-------------+--------------+
|http://www.billbo...|7/17/1965|           34|Don't Just Stand ...|       Patty Duke|Don't Just Stand ...|       1|                    45|           34|             4|
|http://www.billbo...|7/24/1965|           22|Don't Just Stand ...|       Patty Duke|Don't Just Stand ...|       1|                    34|           22|             5|
|http://www.billbo...|7/31/1965|           14|Don't Just Stand ...|       Patty Duke|Don't Just Stand ...|       1|                    22|           14|        

In [15]:
# Verificar se existem linhas duplicadas no DataFrame do ficheiro billboard.csv

duplicates_bb = (
               df_bb
              .groupBy(df_bb.columns)
              .count()
              .filter("count > 1")
              )
duplicates_bb.show()
duplicates_bb.count()

+---+-------+-------------+----+---------+-------+--------+----------------------+-------------+--------------+-----+
|url|week_id|week_position|song|performer|song_id|instance|previous_week_position|peak_position|weeks_on_chart|count|
+---+-------+-------------+----+---------+-------+--------+----------------------+-------------+--------------+-----+
+---+-------+-------------+----+---------+-------+--------+----------------------+-------------+--------------+-----+



0

In [16]:
# Contar o número de missing values (null ou vazios) em cada coluna dos variáveis do ficheiro billboard.csv
missing_counts_bb = df_bb.select([
    F.count(F.when((F.col(c).isNull()) | (F.col(c) == " "), c)).alias(c)
    for c in df_bb.columns
])

missing_counts.show(truncate=False)

+-------+---------+----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+
|song_id|performer|song|spotify_genre|spotify_track_id|spotify_track_preview_url|spotify_track_duration_ms|spotify_track_explicit|spotify_track_album|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|spotify_track_popularity|
+-------+---------+----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+
|0      |0        |0   |0            |0               |0                        |0                        |0          

In [21]:
# Contar o número de valores "NA" em cada coluna dos variáveis do ficheiro billboard.csv
na_values = df_bb.select([
    F.count(
        F.when(F.col(c).cast("string") == "NA", c)
    ).alias(c)
    for c in df_bb.columns
])

na_values.show(truncate=False)

+---+-------+-------------+----+---------+-------+--------+----------------------+-------------+--------------+
|url|week_id|week_position|song|performer|song_id|instance|previous_week_position|peak_position|weeks_on_chart|
+---+-------+-------------+----+---------+-------+--------+----------------------+-------------+--------------+
|0  |0      |0            |0   |0        |0      |0       |31944                 |0            |9             |
+---+-------+-------------+----+---------+-------+--------+----------------------+-------------+--------------+

